# Graph Features Exploration

**Goal:** Characterize current graph morphometry features, run QC and sanity checks, investigate batch/site effects and predictive signal, and deliver a concise conclusion on likely causes of weak pCR prediction signal.

---

## 1. Setup and config

Paths, constants, and random seed for reproducible runs. **NOTE:** These assume that this notebook is located in the DSI Cluster!

In [1]:
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
BASE = Path("/net/projects2/vanguard")

# Dataset Directory
DATASET_DIR = BASE / "MAMA-MIA-syn60868042"

# Directory of per-case morphometry JSONs (one file per case; see ML-Pipeline/pcr_prediction.py)
# FEATURE_DIR = DATASET_DIR / "patient_info_files" # Patient Info JSONs
# FEATURE_DIR = BASE / "centerlines" / "json_files"
FEATURE_DIR = BASE / "vessel_segmentations" / "processed_3D"


# Metadata with patient_id and site/scanner/cohort
METADATA_PATH = DATASET_DIR / "clinical_and_imaging_info.xlsx"

# Labels (pCR): CSV or directory of label JSONs
LABELS_PATH = (
    FEATURE_DIR / "pcr_labels.csv"
)  # e.g. Path("pcr_labels.csv") or BASE / "labels"

# Train/val splits and site proxy (patient_id, fold_idx, dataset, stratum_key)
SPLITS_CSV = Path("splits.csv")

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
EXPECTED_N_CASES = 1500  # approximate cohort size for sanity checks
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
# Print config and basic path checks
print("Config:")
print(f"  FEATURE_DIR      = {FEATURE_DIR}")
print(f"  METADATA_PATH    = {METADATA_PATH}")
print(f"  LABELS_PATH      = {LABELS_PATH}")
print(f"  SPLITS_CSV       = {SPLITS_CSV}")
print(f"  EXPECTED_N_CASES = {EXPECTED_N_CASES}")
print(f"  RANDOM_STATE     = {RANDOM_STATE}")
print()

if FEATURE_DIR.exists():
    n_jsons = len(list(FEATURE_DIR.glob("*.json")))
    print(f"  FEATURE_DIR exists: {n_jsons} .json files")
else:
    print(
        "  FEATURE_DIR does not exist (will need to set or use pre-built features.csv)"
    )
if SPLITS_CSV.exists():
    print(f"  SPLITS_CSV exists: {len(pd.read_csv(SPLITS_CSV))} rows")
else:
    print("  SPLITS_CSV not found")

Config:
  FEATURE_DIR      = /net/projects2/vanguard/vessel_segmentations/processed_3D
  METADATA_PATH    = /net/projects2/vanguard/MAMA-MIA-syn60868042/clinical_and_imaging_info.xlsx
  LABELS_PATH      = /net/projects2/vanguard/vessel_segmentations/processed_3D/pcr_labels.csv
  SPLITS_CSV       = splits.csv
  EXPECTED_N_CASES = 1500
  RANDOM_STATE     = 42

  FEATURE_DIR exists: 1004 .json files
  SPLITS_CSV exists: 1480 rows


## 2. Data loading

Load the feature matrix (from per-case morphometry JSONs or a pre-built CSV), clinical/imaging metadata from `clinical_and_imaging_info.xlsx` (main sheet + `dataset_info` sheet for patient_info-like fields), and optional splits.

**Morphometry filename convention:** `[clinic_id]_[patient_id]_[frame]_vessel_segmentation_morphometry.json` (e.g. `DUKE_001_DUKE_001_0001_vessel_segmentation_morphometry.json`). We parse patient_id (segments 2–3) and frame (5th segment). **One row per frame** (so >4000 rows); within each file we aggregate metrics across vessel components for a fixed feature schema. Global key is `patient_id` only.

In [4]:
# ---------------------------------------------------------------------------
# Load clinical and imaging metadata (clinical_and_imaging_info.xlsx)
# Fields: patient_id, dataset, site, manufacturer, scanner_model, field_strength,
#         pcr, tumor_subtype, age, and others (see data dictionary).
# ---------------------------------------------------------------------------
if METADATA_PATH is not None and METADATA_PATH.exists():
    meta = pd.read_excel(METADATA_PATH, sheet_name=0, engine="openpyxl")
    # Normalize column names (strip whitespace)
    meta.columns = meta.columns.str.strip()
    # Key columns for downstream: id, outcome, batch/site, stratification
    id_col = "patient_id"
    label_col = "pcr"
    site_cols = ["dataset", "site", "manufacturer", "scanner_model", "field_strength"]
    strat_col = "tumor_subtype"
    for c in [id_col, label_col]:
        if c not in meta.columns:
            raise KeyError(f"Metadata missing required column: {c}")
    # Coerce pcr to numeric (0/1); empty or non-numeric -> NaN
    meta[label_col] = pd.to_numeric(meta[label_col], errors="coerce")
    print(f"Metadata: {len(meta)} rows, {len(meta.columns)} columns")
    print(f"  pCR non-null: {meta[label_col].notna().sum()}")
    print(
        f"  Site-related columns present: {[c for c in site_cols if c in meta.columns]}"
    )

    # ---------------------------------------------------------------------------
    # Load dataset_info sheet (patient_info-like fields; join on patient_id)
    # Mirrors patient_info JSON structure (clinical_data, primary_lesion, imaging_data)
    # for exploration and downstream use when features come from morphometry JSONs.
    # ---------------------------------------------------------------------------
    dataset_info = None
    try:
        _di = pd.read_excel(METADATA_PATH, sheet_name="dataset_info", engine="openpyxl")
        _di.columns = _di.columns.str.strip()
        if "patient_id" in _di.columns:
            dataset_info = _di
            print(
                f"  dataset_info sheet: {len(dataset_info)} rows, {len(dataset_info.columns)} columns"
            )
        else:
            print("  dataset_info sheet: no patient_id column; skipping.")
    except Exception as e:
        print(f"  dataset_info sheet: not found or load failed ({e}); skipping.")
else:
    meta = None
    dataset_info = None
    id_col, label_col, site_cols, strat_col = "patient_id", "pcr", [], "tumor_subtype"
    print("METADATA_PATH not set or file missing; skipping metadata load.")

Metadata: 1506 rows, 50 columns
  pCR non-null: 1491
  Site-related columns present: ['dataset', 'site', 'manufacturer', 'scanner_model', 'field_strength']
  dataset_info sheet: 1506 rows, 50 columns


In [5]:
# ---------------------------------------------------------------------------
# Load feature matrix: from pre-built CSV or build from per-case JSONs
# Supports (1) patient-info JSONs: patient_id, clinical_data, primary_lesion, imaging_data
#         (2) morphometry JSONs: vessel/segment metrics (radius, length, etc.)
# ---------------------------------------------------------------------------
FEATURES_CSV = None  # Set to Path("features.csv") if you have a pre-built table


# Morphometry filename: [clinic_id]_[patient_id]_[frame]_vessel_segmentation_morphometry
_MIN_PARTS_FOR_FRAME_FORMAT = 5
_MIN_PARTS_FOR_PATIENT_ID = 2


def parse_morphometry_filename(stem: str) -> tuple[str, int]:
    """Parse morphometry JSON stem: [clinic_id]_[patient_id]_[frame]_vessel_segmentation_morphometry.

    Returns (patient_id, frame). E.g. DUKE_001_DUKE_001_0001 -> ('DUKE_001', 1).
    """
    parts = stem.split("_")
    try:
        idx = parts.index("vessel")
    except ValueError:
        idx = len(parts)
    if idx >= _MIN_PARTS_FOR_FRAME_FORMAT:
        patient_id = "_".join(parts[2:4])
        try:
            frame = int(parts[4])
        except (ValueError, IndexError):
            frame = 0
    else:
        patient_id = (
            "_".join(parts[:_MIN_PARTS_FOR_PATIENT_ID])
            if len(parts) >= _MIN_PARTS_FOR_PATIENT_ID
            else ""
        )
        frame = 0
    return (patient_id, frame)


def _flatten_dict(d: dict, prefix: str = "") -> dict:
    """Recursively flatten nested dict to keys like prefix__key__subkey with scalar values."""
    out = {}
    for k, v in d.items():
        key = f"{prefix}__{k}" if prefix else k
        if isinstance(v, dict):
            out.update(_flatten_dict(v, key))
        elif not isinstance(v, list):
            out[key] = v
    return out


def _scalar_to_value(x: object) -> object:
    """Convert scalar to numeric where possible for feature table."""
    if isinstance(x, int | float) and not isinstance(x, bool):
        return float(x) if np.isfinite(x) else np.nan
    if isinstance(x, bool | np.bool_):
        return int(x)
    if isinstance(x, str):
        if x.strip() == "":
            return np.nan
        try:
            return float(x.strip())
        except ValueError:
            return x
    return x


def _is_patient_info_schema(data: dict) -> bool:
    """True if JSON has patient_id and clinical_data / primary_lesion / imaging_data structure."""
    if not isinstance(data, dict) or "patient_id" not in data:
        return False
    return any(k in data for k in ("clinical_data", "primary_lesion", "imaging_data"))


def build_features_from_patient_info_jsons(feature_dir: Path) -> pd.DataFrame:
    """Build one row per case from patient-info JSONs (patient_id, clinical_data, primary_lesion, imaging_data)."""
    rows = []
    for p in sorted(feature_dir.glob("*.json")):
        try:
            data = json.loads(p.read_text())
        except Exception as e:
            logging.warning("Skipping %s: %s", p, e)
            continue
        if not _is_patient_info_schema(data):
            continue
        patient_id = data.get("patient_id")
        if not patient_id:
            continue
        flat = _flatten_dict(data)
        row = {"patient_id": patient_id}
        for k, v in flat.items():
            if k == "patient_id":
                continue
            row[k] = _scalar_to_value(v)
        rows.append(row)
    if not rows:
        return pd.DataFrame(columns=["patient_id"])
    out = pd.DataFrame(rows)
    cols_no_id = [c for c in out.columns if c != "patient_id"]
    if cols_no_id:
        const = [c for c in cols_no_id if out[c].eq(out[c].iloc[0]).all()]
        out = out.drop(columns=const, errors="ignore")
    return out


def build_features_from_morphometry_jsons(feature_dir: Path) -> pd.DataFrame:
    """Build one row per frame from morphometry JSONs (>4000 rows).

    Filename format: [clinic_id]_[patient_id]_[frame]_vessel_segmentation_morphometry.json
    - Parses patient_id and frame from each filename (global key: patient_id only).
    - Within each file: aggregates metrics across all vessel components for a fixed schema.
    """
    from collections import defaultdict

    def to_num(x: object) -> float | None:
        if isinstance(x, bool | np.bool_):
            return float(x)
        if isinstance(x, int | float | np.integer | np.floating):
            return float(x)
        if isinstance(x, str):
            try:
                return float(x.strip())
            except Exception:
                return None
        return None

    # One row per file (per frame) with fixed schema (aggregate across vessels)
    file_rows = []
    for p in sorted(feature_dir.glob("*.json")):
        try:
            data = json.loads(p.read_text())
        except Exception as e:
            logging.warning("Skipping %s: %s", p, e)
            continue
        stem = p.stem
        patient_id, frame = parse_morphometry_filename(stem)
        global_vals = defaultdict(list)
        if isinstance(data, dict):
            for _, group in data.items():
                if not isinstance(group, dict):
                    continue
                for _vessel_name, items in group.items():
                    if not isinstance(items, list):
                        continue
                    for item in items:
                        if not isinstance(item, dict):
                            continue
                        for k, v in item.items():
                            if isinstance(v, dict):
                                for kk, vv in v.items():
                                    if isinstance(vv, list):
                                        nums = [
                                            to_num(t)
                                            for t in vv
                                            if to_num(t) is not None
                                        ]
                                        if nums:
                                            global_vals[f"{k}__{kk}"].extend(nums)
                                    else:
                                        n = to_num(vv)
                                        if n is not None:
                                            global_vals[f"{k}__{kk}"].append(n)
                            elif isinstance(v, list):
                                nums = [to_num(t) for t in v if to_num(t) is not None]
                                if nums:
                                    global_vals[k].extend(nums)
                            else:
                                n = to_num(v)
                                if n is not None:
                                    global_vals[k].append(n)
        feats = {"patient_id": patient_id, "frame": frame}
        for fld, vals in global_vals.items():
            if not vals:
                continue
            feats[f"{fld}__mean"] = float(np.mean(vals))
            feats[f"{fld}__std"] = float(np.std(vals)) if len(vals) > 1 else 0.0
            feats[f"{fld}__min"] = float(np.min(vals))
            feats[f"{fld}__max"] = float(np.max(vals))
            feats[f"{fld}__count"] = float(len(vals))
        file_rows.append(feats)

    if not file_rows:
        return pd.DataFrame(columns=["patient_id", "frame"])

    out = pd.DataFrame(file_rows)
    id_cols = ["patient_id", "frame"]
    cols_no_id = [c for c in out.columns if c not in id_cols]
    if cols_no_id:
        const = [c for c in cols_no_id if out[c].eq(out[c].iloc[0]).all()]
        out = out.drop(columns=const, errors="ignore")
    return out


if FEATURES_CSV is not None and Path(FEATURES_CSV).exists():
    features_df = pd.read_csv(FEATURES_CSV)
    if "patient_id" not in features_df.columns and "case_id" in features_df.columns:
        features_df = features_df.rename(columns={"case_id": "patient_id"})
    print(
        f"Features: loaded from {FEATURES_CSV} -> {len(features_df)} rows, {len(features_df.columns)} columns"
    )
elif FEATURE_DIR.exists():
    # Detect schema from first valid JSON
    json_paths = sorted(FEATURE_DIR.glob("*.json"))
    schema = "morphometry"
    for path in json_paths[:5]:
        try:
            sample = json.loads(path.read_text())
            if _is_patient_info_schema(sample):
                schema = "patient_info"
            break
        except Exception as e:
            logging.warning("Could not read sample JSON to detect schema: %s", e)
            continue
    if schema == "patient_info":
        features_df = build_features_from_patient_info_jsons(FEATURE_DIR)
        print(
            f"Features: built from patient-info JSONs in {FEATURE_DIR} -> {len(features_df)} rows, {len(features_df.columns)} columns"
        )
    else:
        features_df = build_features_from_morphometry_jsons(FEATURE_DIR)
        print(
            f"Features: built from morphometry JSONs in {FEATURE_DIR} -> {len(features_df)} rows, {len(features_df.columns)} columns"
        )
else:
    features_df = pd.DataFrame(columns=["patient_id"])
    print("No feature directory or pre-built CSV; features_df is empty.")

Features: built from morphometry JSONs in /net/projects2/vanguard/vessel_segmentations/processed_3D -> 1004 rows, 117 columns


In [6]:
# ---------------------------------------------------------------------------
# Merge features with metadata (patient_id) and optional splits
# ---------------------------------------------------------------------------
if meta is not None and len(features_df) > 0 and "patient_id" in features_df.columns:
    merged = features_df.merge(
        meta[
            [c for c in [id_col, label_col, strat_col] + site_cols if c in meta.columns]
        ],
        on="patient_id",
        how="inner",
    )
    if SPLITS_CSV.exists():
        splits_df = pd.read_csv(SPLITS_CSV)
        splits_sub = splits_df[["patient_id", "fold_idx", "stratum_key"]]
        merged = merged.merge(
            splits_sub,
            on="patient_id",
            how="left",
        )
    if dataset_info is not None and len(merged) > 0:
        di_cols_to_add = [
            c
            for c in dataset_info.columns
            if c != "patient_id" and c not in merged.columns
        ]
        if di_cols_to_add:
            di_subset = dataset_info[["patient_id"] + di_cols_to_add]
            merged = merged.merge(
                di_subset,
                on="patient_id",
                how="left",
                suffixes=("", "_di"),
            )
            if "patient_id_di" in merged.columns:
                merged = merged.drop(columns=["patient_id_di"], errors="ignore")
            print(f"  dataset_info columns added: {len(di_cols_to_add)}")
    print(f"Merged (features + metadata): {len(merged)} rows")
    print(f"  With non-missing pCR:     {merged[label_col].notna().sum()}")
    if "dataset" in merged.columns:
        print(
            f"  dataset value_counts:\n{merged['dataset'].value_counts().to_string()}"
        )
else:
    merged = features_df.copy() if len(features_df) > 0 else pd.DataFrame()
    if (
        meta is not None
        and "patient_id" in meta.columns
        and "patient_id" in merged.columns
    ):
        merged = merged.merge(meta, on="patient_id", how="left")
    if dataset_info is not None and len(merged) > 0 and "patient_id" in merged.columns:
        di_cols_to_add = [
            c
            for c in dataset_info.columns
            if c != "patient_id" and c not in merged.columns
        ]
        if di_cols_to_add:
            di_subset = dataset_info[["patient_id"] + di_cols_to_add]
            merged = merged.merge(
                di_subset,
                on="patient_id",
                how="left",
                suffixes=("", "_di"),
            )
            if "patient_id_di" in merged.columns:
                merged = merged.drop(columns=["patient_id_di"], errors="ignore")
    print("Merged: no merge performed (missing features or metadata).")

  dataset_info columns added: 42
Merged (features + metadata): 1004 rows
  With non-missing pCR:     984
  dataset value_counts:
dataset
ISPY2    502
DUKE     467
ISPY1     35


In [13]:
# ---------------------------------------------------------------------------
# Validate patient_id matching (features vs metadata)
# ---------------------------------------------------------------------------
if meta is not None and len(features_df) > 0 and "patient_id" in features_df.columns:
    meta_ids = set(meta[id_col].astype(str).str.strip())
    feature_ids = set(features_df["patient_id"].astype(str).str.strip())
    in_both = feature_ids & meta_ids
    only_in_features = feature_ids - meta_ids
    only_in_metadata = meta_ids - feature_ids
    print("Patient ID matching (features vs metadata):")
    print(f"  Unique patient_ids in metadata: {len(meta_ids)}")
    print(f"  Unique patient_ids in features: {len(feature_ids)}")
    print(f"  In both (matched):     {len(in_both)}")
    MAX_IDS_TO_PRINT = 20
    print(f"  Only in features:     {len(only_in_features)}  (no metadata row)")
    print(f"  Only in metadata:     {len(only_in_metadata)}  (no feature row)")
    if only_in_features and len(only_in_features) <= MAX_IDS_TO_PRINT:
        print(f"  Only in features IDs:  {sorted(only_in_features)}")
    elif only_in_features:
        print(
            f"  Only in features IDs:  {sorted(only_in_features)[:10]} ... ({len(only_in_features)} total)"
        )
    if only_in_metadata and len(only_in_metadata) <= MAX_IDS_TO_PRINT:
        print(f"  Only in metadata IDs:  {sorted(only_in_metadata)}")
    elif only_in_metadata:
        print(
            f"  Only in metadata IDs:  {sorted(only_in_metadata)[:10]} ... ({len(only_in_metadata)} total)"
        )
else:
    print("Skipping patient_id validation (no metadata or features).")

Patient ID matching (features vs metadata):
  Unique patient_ids in metadata: 1506
  Unique patient_ids in features: 208
  In both (matched):     208
  Only in features:     0  (no metadata row)
  Only in metadata:     1298  (no feature row)
  Only in metadata IDs:  ['DUKE_005', 'DUKE_009', 'DUKE_010', 'DUKE_019', 'DUKE_021', 'DUKE_022', 'DUKE_028', 'DUKE_032', 'DUKE_040', 'DUKE_041'] ... (1298 total)


In [14]:
# Summary: sample sizes for downstream sections
n_cases = len(merged)
skip_cols = {id_col, "patient_id", "frame", label_col, strat_col, "fold_idx"} | set(
    site_cols
)
feature_cols = [
    c
    for c in merged.columns
    if c not in skip_cols and pd.api.types.is_numeric_dtype(merged[c])
]
n_with_label = merged[label_col].notna().sum()

print(f"Cases in merged table: {n_cases}")
print(f"Cases with non-missing pCR: {n_with_label}")
print(f"Numeric feature columns (excl. id/label/site): {len(feature_cols)}")
print("\nAll columns:")
for c in merged.columns:
    print(f"  {c}")

Cases in merged table: 1004
Cases with non-missing pCR: 984
Numeric feature columns (excl. id/label/site): 146

All columns:
  patient_id
  frame
  segment__start__mean
  segment__start__std
  segment__start__min
  segment__start__max
  segment__start__count
  segment__end__mean
  segment__end__std
  segment__end__min
  segment__end__max
  segment__end__count
  radius__mean__mean
  radius__mean__std
  radius__mean__min
  radius__mean__max
  radius__mean__count
  radius__sd__mean
  radius__sd__std
  radius__sd__min
  radius__sd__max
  radius__sd__count
  radius__median__mean
  radius__median__std
  radius__median__min
  radius__median__max
  radius__median__count
  radius__min__mean
  radius__min__std
  radius__min__min
  radius__min__max
  radius__min__count
  radius__q1__mean
  radius__q1__std
  radius__q1__min
  radius__q1__max
  radius__q1__count
  radius__q3__mean
  radius__q3__std
  radius__q3__min
  radius__q3__max
  radius__q3__count
  radius__max__mean
  radius__max__std
  radi

## 3. Human-readable feature list

Features may come from **patient-info JSONs** (clinical_data, primary_lesion, imaging_data) or from **morphometry JSONs** (vessel/segment metrics). For morphometry: radius stats, length, tortuosity, volume, curvature stats, bifurcation angles; flattened with `__mean`, `__std`, `__min`, `__max`, `__count`. Below we group the actual columns in this run by feature type.

In [9]:
# Feature type definitions (pipeline schema + patient-info JSON schema)
FEATURE_TYPE_DEFINITIONS = {
    # Patient-info JSON (clinical_data, primary_lesion, imaging_data)
    "clinical_data": "From patient-info JSON: age, menopausal_status, breast_density, etc.",
    "primary_lesion": "From patient-info JSON: pcr, tumor_subtype, breast_coordinates (x_min, x_max, ...).",
    "imaging_data": "From patient-info JSON: dataset, site, scanner_manufacturer, field_strength, echo_time, repetition_time, bilateral.",
    # Morphometry (vessel/segment)
    "radius": "Vessel radius at skeleton points (from distance transform); stats: mean, sd, median, min, q1, q3, max. Expect > 0.",
    "length": "Path length of vessel segment (sum of edge lengths). Expect > 0.",
    "tortuosity": "Path length / straight-line distance; ≥ 1.",
    "curvature": "Angular change per unit path (rad or deg). Expect ≥ 0.",
    "angle": "Bifurcation opening angles between branch pairs. Expect [0, 180] degrees.",
    "volume": "Segment volume (e.g. π·r²·length). Expect > 0.",
    "count": "Number of segments or points (__count suffix).",
    "other": "Other numeric or categorical fields.",
}

In [10]:
# Map keywords (lowercase) to feature type for grouping (patient-info prefixes first)
KEYWORD_TO_TYPE = {
    "clinical_data": "clinical_data",
    "primary_lesion": "primary_lesion",
    "imaging_data": "imaging_data",
    "radius": "radius",
    "length": "length",
    "tortuosity": "tortuosity",
    "curvature": "curvature",
    "angle": "angle",
    "volume": "volume",
    "area": "volume",  # treat area as volume-like
    "count": "count",
}


def classify_feature_column(name: str) -> str:
    """Return feature type (radius, length, tortuosity, etc.) from column name."""
    n = name.lower()
    for kw, ftype in KEYWORD_TO_TYPE.items():
        if kw in n:
            return ftype
    return "other"


# Build grouped list from actual feature columns
if feature_cols:
    grouped = {}
    for c in feature_cols:
        ftype = classify_feature_column(c)
        grouped.setdefault(ftype, []).append(c)
    for ftype in [
        "clinical_data",
        "primary_lesion",
        "imaging_data",
        "radius",
        "length",
        "tortuosity",
        "curvature",
        "angle",
        "volume",
        "count",
        "other",
    ]:
        if ftype in grouped:
            grouped[ftype] = sorted(grouped[ftype])
    feature_groups = grouped
else:
    feature_groups = {}

In [ ]:
# Display: definitions table + column counts and sample column names per type
print("Feature type definitions (from pipeline)\n")
for ftype, desc in FEATURE_TYPE_DEFINITIONS.items():
    print(f"  {ftype}: {desc}")

print("\nColumn counts and example columns per type (this run):")
for ftype in [
    "clinical_data",
    "primary_lesion",
    "imaging_data",
    "radius",
    "length",
    "tortuosity",
    "curvature",
    "angle",
    "volume",
    "count",
    "other",
]:
    cols = feature_groups.get(ftype, [])
    n = len(cols)
    examples = cols[:3] if n else []
    print(f"  {ftype}: {n} columns  e.g. {examples}")

Feature type definitions (from pipeline)

  clinical_data: From patient-info JSON: age, menopausal_status, breast_density, etc.

  primary_lesion: From patient-info JSON: pcr, tumor_subtype, breast_coordinates (x_min, x_max, ...).

  imaging_data: From patient-info JSON: dataset, site, scanner_manufacturer, field_strength, echo_time, repetition_time, bilateral.

  radius: Vessel radius at skeleton points (from distance transform); stats: mean, sd, median, min, q1, q3, max. Expect > 0.

  length: Path length of vessel segment (sum of edge lengths). Expect > 0.

  tortuosity: Path length / straight-line distance; ≥ 1.

  curvature: Angular change per unit path (rad or deg). Expect ≥ 0.

  angle: Bifurcation opening angles between branch pairs. Expect [0, 180] degrees.

  volume: Segment volume (e.g. π·r²·length). Expect > 0.

  count: Number of segments or points (__count suffix).

  other: Other numeric or categorical fields.


Column counts and example columns per type (this run):
  cl

In [12]:
# Optional: full list of feature column names (uncomment to inspect)
# for ftype, cols in feature_groups.items():
#     print(f"\n--- {ftype} ({len(cols)}) ---")
#     for c in cols:
#         print(f"  {c}")